## Imports

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.inspection import permutation_importance

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    make_scorer
)
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt



## Daten laden

In [ ]:
DATA_ROOT = Path("../data")
FEAT_PATH = DATA_ROOT / "processed" / "features" / "features_5s_2p5s_clean.parquet"
assert FEAT_PATH.exists(), f"Nicht gefunden: {FEAT_PATH.resolve()}"

df = pd.read_parquet(FEAT_PATH)
print("Loaded:", df.shape)
print(df["label"].value_counts())
df.head()


## Feature/Labels/Groups definieren

In [ ]:
# Zielvariable
y = df["label"].astype(int).values

groups = (df["source"].astype(str) + "_" + df["record"].astype(str)).values

meta_cols = ["source", "record", "start_s", "label", "npz_file"]
feature_cols = [c for c in df.columns if c not in meta_cols]

X = df[feature_cols].values

print("X shape:", X.shape, "y shape:", y.shape)
print("n_features:", len(feature_cols))
print("Example features:", feature_cols[:20])


## Train/Test Split (GroupShuffleSplit)

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

groups_train = groups[train_idx]
groups_test  = groups[test_idx]

print("Train size:", X_train.shape, "Test size:", X_test.shape)
print("Unique records train:", len(np.unique(groups_train)))
print("Unique records test :", len(np.unique(groups_test)))
print("Class balance train:", np.bincount(y_train))
print("Class balance test :", np.bincount(y_test))


## Train/Test Split speichern

In [ ]:
OUT_DIR = DATA_ROOT / "processed" / "models"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.Series(np.unique(groups_train)).to_csv(OUT_DIR / "train_records.csv", index=False, header=["record"])
pd.Series(np.unique(groups_test)).to_csv(OUT_DIR / "test_records.csv", index=False, header=["record"])
print("Saved split lists to:", OUT_DIR.resolve())


## Random Forest trainieren (Cross Validation & Random Search)

In [ ]:
sensitivity_scorer = make_scorer(lambda yt, yp: confusion_matrix(yt, yp).ravel()[3] / (confusion_matrix(yt, yp).ravel()[3] + confusion_matrix(yt, yp).ravel()[2] + 1e-12))
cv = GroupKFold(n_splits=5)

param_distributions = {
    "n_estimators": [200, 300, 500, 800],
    "max_depth": [None, 10, 20, 30],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": ["sqrt", "log2", None]
}

rf_base = RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_distributions,
    n_iter=20,                 
    scoring="recall",         
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

search.fit(X_train, y_train, groups=groups_train)

print("Best params:", search.best_params_)
print("Best CV sensitivity (recall):", search.best_score_)


## Bestes Modell auf Test-Set evaluieren

In [ ]:
best_rf = search.best_estimator_

y_proba = best_rf.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)   

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
specificity = tn / (tn + fp) if (tn + fp) else 0.0

print("Confusion Matrix:\n", cm)
print(f"Sensitivity: {sensitivity:.3f}")
print(f"Specificity: {specificity:.3f}")


## Modell speichern

In [ ]:
model_path = OUT_DIR / "rf_tuned_groupcv.joblib"
joblib.dump({"model": best_rf, "feature_cols": feature_cols, "best_params": search.best_params_}, model_path)
print(model_path)
